# 太陽能板瑕疵檢測：兩階段流程（VLMtrain v3）

**做法**：偵測模型只負責「找出每個模組」，再對每個模組逐一判斷是否異常。

| 類別 | 判斷方式 | 原因 |
|---|---|---|
| solar_panel | YOLO 分割偵測，不分類別做 NMS、去碎框 | 位置穩定 |
| hotspot（IR） | 模組亮度/邊緣統計，與同一張圖的其他模組比較 | 「比鄰居更熱」是物理特徵，整組留出驗證 AUC 0.86~1.0 |
| crack（EO） | 模組裁切圖的 ResNet18 特徵 + 邏輯迴歸 | 獨立的 crack 樣本極少（同一模組連拍），效果不穩，視為賭注 |

**執行方式**：重開 kernel 後，從上到下依序執行。已經做完的步驟（偵測模型權重、特徵檔）會自動沿用，不會重做。
想重訓偵測模型，刪掉 `outputs/runs/` 底下對應的資料夾即可。

**上傳前請先看第 10 步「框品質」**：測試集的 IoU 要接近驗證集，否則 solar_panel 與 hotspot 都拿不到分（前幾次上傳 solar_panel 只有 0.5 左右，驗證集卻有 0.9）。


## 0. 安裝套件

In [1]:
# 第一次執行、或新環境才需要；已經裝過的套件會自動略過（裝完不必重啟 kernel）
%pip install -q shapely scipy scikit-learn bert-score transformers torchvision


Note: you may need to restart the kernel to use updated packages.


## 1. 設定

In [2]:
# ==========================================
# 0. 共用設定（重開 kernel 後，從這一格開始往下「依序」執行）
# ==========================================
import json, time, math, shutil, random, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import cv2

DATA_ROOT = Path(r"C:\Users\Sharon\Downloads\VLPaicup")
TRAIN_ROOT = DATA_ROOT / "9_16_train_set" / "train"
TEST_BASE = DATA_ROOT / "9_30_test_set"
KIT_ROOT = DATA_ROOT / "evaluation_kit" / "author-kit_1012"
GROUP_FILE = DATA_ROOT / "SolarDefectDataset_supplement" / "train_groups.csv"
OUT = DATA_ROOT / "outputs"
RUNS = OUT / "runs"
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = "xpu:0"

# 偵測模型：只負責「找出每個模組的位置」。已經訓練完的權重（runs/eo_v2、runs/ir_v2）會直接沿用。
# 想重新訓練：刪掉對應的 runs 資料夾。EO 用 imgsz=768、batch=2 是為了避免 XPU 記憶體不足。
CFG = {
    "EO": dict(defect="crack", yaml=OUT / "yolo_eo" / "data.yaml", model="yolo11s-seg.pt",
               imgsz=768, batch=2, epochs=80, name="eo_v2"),
    "IR": dict(defect="hotspot", yaml=OUT / "yolo_ir" / "data.yaml", model="yolo11s-seg.pt",
               imgsz=640, batch=8, epochs=100, name="ir_v2"),
}


def val_weights(modality):
    """只用 train split 訓練的那一版（驗證用）"""
    return RUNS / CFG[modality]["name"] / "weights" / "best.pt"


def best_weights(modality):
    """推論用：有「全資料重訓」的權重就用它，否則用 val 版"""
    final = RUNS / (CFG[modality]["name"] + "_final") / "weights" / "last.pt"
    return final if final.exists() else val_weights(modality)


# ---- 兩階段（偵測模組 -> 逐模組分類）的參數 ----
# EO 推論解析度：實測 768 的框品質 0.947，1536 反而降到 0.888（同一批測試圖），所以用 768。
# IR 輸入一律轉灰階（見 load_source），否則測試集約 25% 的偽彩色圖找不到模組。
INFER_IMGSZ = {"EO": 768, "IR": 640}
INFER_BATCH = {"EO": 2, "IR": 8}
SCALE = {"IR": 1.0, "EO": 0.5}   # 算模組特徵時影像縮放比例（EO 太大，縮一半加速）
T_PANEL_V3 = 0.5                 # 偵測框信心度門檻（不分類別）
# hotspot 的判斷方式：「每張圖取分數最高的前 HOT_K 個模組，且分數 >= HOT_TAU」。
# 不用固定機率門檻，因為分類器的機率尺度換組後會縮放（整組留出：G02 正樣本機率中位數只有 0.0003，固定 0.02 會標不出來）。
# 端到端驗證（G04，真實偵測框）：固定 0.01 -> F1 0.87；前 1 名 -> 0.77（召回 208/216）；前 2 名 -> 0.49（每張圖多標一個就多一倍誤判）。
# 所以取前 1 名，地板值 1e-3（G02 標註框 F1 0.57、G03 0.50）。依據：428 張有 hotspot 的圖中 412 張只有 1 個。
HOT_K = 1
HOT_TAU = 1e-3
USE_EO_CRACK = True              # 是否在 EO 輸出 crack 預測
CRACK_FRAC = 0.04                # EO 測試模組中，機率最高的這個比例標成 crack（偏向召回，讓每個成因類別都有機會配對到）

for m, c in CFG.items():
    print(m, "| imgsz:", c["imgsz"], "| batch:", c["batch"], "| epochs:", c["epochs"],
          "| 已有權重:", val_weights(m).exists())


EO | imgsz: 768 | batch: 2 | epochs: 80 | 已有權重: True
IR | imgsz: 640 | batch: 8 | epochs: 100 | 已有權重: True


## 2. 資料準備（切分 + YOLO 格式）

In [3]:
# ==========================================
# 1. 資料準備：訓練/驗證切分 + YOLO 格式資料集（已經存在就自動略過）
#    切分以 group 為單位：EO 留出 G01、IR 留出 G04 當驗證組
# ==========================================
from PIL import Image

SPLIT_FILE = OUT / "train_val_split.csv"
if not SPLIT_FILE.exists():
    g = pd.read_csv(GROUP_FILE)
    g["EO_split"] = np.where(g["group"] == "G01", "val", "train")
    g["IR_split"] = np.where(g["group"] == "G04", "val", "train")
    g.to_csv(SPLIT_FILE, index=False, encoding="utf-8-sig")
    print("已建立：", SPLIT_FILE)
split_df = pd.read_csv(SPLIT_FILE)
print("影格數：", len(split_df))


def convert_polygon(shape, w, h):
    pts = shape.get("points", [])
    if len(pts) < 3:
        raise ValueError("Polygon has fewer than 3 points.")
    out = []
    for x, y in pts:
        x, y = float(x), float(y)
        if not (math.isfinite(x) and math.isfinite(y)):
            raise ValueError("invalid coordinates")
        if x < -1 or x > w + 1 or y < -1 or y > h + 1:
            raise ValueError(f"coordinate out of bounds: {x},{y}")
        out += [min(max(x, 0), w) / w, min(max(y, 0), h) / h]
    if len(out) >= 6 and out[0] == out[-2] and out[1] == out[-1]:
        out = out[:-2]
    if len(out) < 6:
        raise ValueError("Polygon has fewer than 3 unique points.")
    return out


def build_yolo(modality):
    cfg = CFG[modality]
    yaml_path = cfg["yaml"]
    if yaml_path.exists():
        print(f"[{modality}] YOLO 資料集已存在，略過：{yaml_path.parent}")
        return
    dataset_dir = yaml_path.parent
    classes = {"solar_panel": 0, cfg["defect"]: 1}
    for sub in ("images", "labels"):
        if (dataset_dir / sub).exists():
            shutil.rmtree(dataset_dir / sub)
        for sp in ("train", "val"):
            (dataset_dir / sub / sp).mkdir(parents=True, exist_ok=True)
    errors, n = [], {"train": 0, "val": 0}
    for _, row in split_df.iterrows():
        fid, sp = row["frame_id"], row[f"{modality}_split"]
        try:
            img_path = TRAIN_ROOT / modality / "image" / f"{fid}.jpg"
            data = json.loads((TRAIN_ROOT / modality / "label" / f"{fid}.json").read_text(encoding="utf-8"))
            w, h = data["imageWidth"], data["imageHeight"]
            with Image.open(img_path) as im:
                if im.size != (w, h):
                    raise ValueError("image size mismatch")
            lines = []
            for s in data.get("shapes", []):
                if s["label"] not in classes:
                    raise ValueError(f"unknown label {s['label']}")
                coords = convert_polygon(s, w, h)
                lines.append(str(classes[s["label"]]) + " " + " ".join(f"{v:.6f}" for v in coords))
            shutil.copy2(img_path, dataset_dir / "images" / sp / img_path.name)
            (dataset_dir / "labels" / sp / f"{fid}.txt").write_text("\n".join(lines), encoding="utf-8")
            n[sp] += 1
        except Exception as e:
            errors.append(f"{fid}: {e}")
    yaml_path.write_text(
        f"path: '{dataset_dir.as_posix()}'\ntrain: images/train\nval: images/val\n\nnames:\n"
        + "".join(f"  {i}: {name}\n" for name, i in classes.items()), encoding="utf-8")
    print(f"[{modality}] 完成：train {n['train']} / val {n['val']}，錯誤 {len(errors)}")
    for e in errors[:5]:
        print("   ", e)


for m in CFG:
    build_yolo(m)


影格數： 1046
[EO] YOLO 資料集已存在，略過：C:\Users\Sharon\Downloads\VLPaicup\outputs\yolo_eo
[IR] YOLO 資料集已存在，略過：C:\Users\Sharon\Downloads\VLPaicup\outputs\yolo_ir


## 3. 偵測模型訓練函式

In [4]:
# ==========================================
# V2-A2. 訓練函式（可中斷後續跑：重跑這一格會自動 resume）
# ==========================================
import time
import torch
from ultralytics import YOLO

assert torch.xpu.is_available(), "Intel XPU is unavailable"

FORCE_RETRAIN = False  # True = 忽略已有的結果，整個重訓


def make_all_yaml(modality):
    """train + val 合併訓練用的 yaml（最後一次全資料重訓用）"""
    c = CFG[modality]
    base = c["yaml"].parent
    path = base / "data_all.yaml"
    path.write_text(
        f"path: '{base.as_posix()}'\n"
        "train:\n  - images/train\n  - images/val\n"
        "val: images/val\n\n"
        f"names:\n  0: solar_panel\n  1: {c['defect']}\n",
        encoding="utf-8",
    )
    return path


def train_one(modality, final=False):
    c = CFG[modality]
    name = c["name"] + ("_final" if final else "")
    run_dir = RUNS / name
    done_flag = run_dir / "DONE"
    last = run_dir / "weights" / "last.pt"

    if done_flag.exists() and not FORCE_RETRAIN:
        print(f"[{modality}] {name} 已訓練完成，略過（FORCE_RETRAIN=True 可重訓）")
        return

    t0 = time.time()
    if last.exists() and not FORCE_RETRAIN:
        print(f"[{modality}] 從 {last} 接續訓練")
        YOLO(str(last)).train(resume=True)
    else:
        data_yaml = make_all_yaml(modality) if final else c["yaml"]
        model = YOLO(c["model"])
        model.train(
            data=str(data_yaml),
            epochs=c["epochs"],
            imgsz=c["imgsz"],
            batch=c["batch"],
            device=DEVICE,
            workers=0,
            amp=False,
            cache="ram",          # 把縮放後的圖放進記憶體，省掉每個 epoch 重新解碼 JPEG
            val=not final,        # 最後全資料重訓時 val 已包含在訓練集，不用再驗證
            patience=30,
            cos_lr=True,
            close_mosaic=10,
            scale=0.3,
            fliplr=0.5,
            flipud=0.0,
            seed=0,
            plots=False,
            project=str(RUNS),
            name=name,
            exist_ok=True,
            verbose=True,
        )

    run_dir.mkdir(parents=True, exist_ok=True)
    done_flag.write_text(f"finished in {(time.time() - t0) / 60:.1f} min\n", encoding="utf-8")
    print(f"[{modality}] {name} 完成，用時 {(time.time() - t0) / 60:.1f} 分鐘")


## 3a. EO 偵測模型（已訓練完會略過）

In [5]:
# EO（可見光）偵測模型。已訓練完成（有 DONE 標記）會直接略過；中斷過會自動接續。
train_one("EO")
print("EO 權重：", val_weights("EO"))


[EO] eo_v2 已訓練完成，略過（FORCE_RETRAIN=True 可重訓）
EO 權重： C:\Users\Sharon\Downloads\VLPaicup\outputs\runs\eo_v2\weights\best.pt


## 3b. IR 偵測模型（已訓練完會略過）

In [6]:
# IR（紅外線）偵測模型。同上。
train_one("IR")
print("IR 權重：", val_weights("IR"))


[IR] ir_v2 已訓練完成，略過（FORCE_RETRAIN=True 可重訓）
IR 權重： C:\Users\Sharon\Downloads\VLPaicup\outputs\runs\ir_v2\weights\best.pt


## 4. 工具函式：本機評分（官方公式）、多邊形整理、批次預測

In [7]:
# ==========================================
# V2-B. 工具函式：本機評分（依官方 README 第 5、6 節）、多邊形整理、批次預測
# 需要：pip install shapely
# ==========================================
import json
import cv2
import numpy as np
import pandas as pd
from shapely.geometry import Polygon, box as shapely_box
from shapely.ops import unary_union

# ---------- 幾何 ----------

def to_poly(points):
    """points -> shapely 多邊形。2 點視為矩形；不合法者以 buffer(0) 修復（與官方相同）"""
    pts = [(float(x), float(y)) for x, y in points]
    if len(pts) == 2:
        (x1, y1), (x2, y2) = pts
        return shapely_box(min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2))
    if len(pts) < 3:
        return Polygon()
    poly = Polygon(pts)
    if not poly.is_valid:
        poly = poly.buffer(0)
    return poly


def clip_points(pts, w, h):
    pts = np.asarray(pts, dtype=np.float32).reshape(-1, 2).copy()
    pts[:, 0] = np.clip(pts[:, 0], 0, w)
    pts[:, 1] = np.clip(pts[:, 1], 0, h)
    return pts


def limit_vertices(pts, max_pts=100):
    """頂點超過官方上限（100）時，逐步加大容許誤差直到符合"""
    pts = np.asarray(pts, dtype=np.float32).reshape(-1, 2)
    eps = 0.5
    while len(pts) > max_pts:
        simp = cv2.approxPolyDP(pts.reshape(-1, 1, 2), eps, True).reshape(-1, 2)
        if len(simp) >= 3:
            pts = simp
        eps *= 1.5
        if eps > 1e4:
            break
    return pts


def regularize(pts, eps_frac=0.015):
    """模組是凸四邊形：先取凸包，再用 approxPolyDP 壓成少數頂點，去掉遮罩的鋸齒邊"""
    pts = np.asarray(pts, dtype=np.float32).reshape(-1, 2)
    if len(pts) < 3:
        return None
    hull = cv2.convexHull(pts).reshape(-1, 2)
    if len(hull) < 3:
        return None
    peri = cv2.arcLength(hull.reshape(-1, 1, 2), True)
    approx = cv2.approxPolyDP(hull.reshape(-1, 1, 2), eps_frac * peri, True).reshape(-1, 2)
    return approx if len(approx) >= 3 else hull


def final_points(pts, use_reg, w, h):
    """輸出到提交檔的頂點：整理 -> 限制 100 點 -> 裁到影像範圍 -> 四捨五入。失敗回傳 None"""
    pts = np.asarray(pts, dtype=np.float32).reshape(-1, 2)
    if use_reg:
        reg = regularize(pts)
        if reg is not None:
            pts = reg
    pts = limit_vertices(pts)
    if len(pts) < 3:
        return None
    pts = clip_points(pts, w, h)
    return [[round(float(x), 2), round(float(y), 2)] for x, y in pts]


def box_iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


# ---------- 官方定位分數 F1_TIoU（README 第 5 節）----------

def class_scores(gt_per_img, pred_per_img):
    """單一類別。回傳 (sum_r, n_gt, sum_s, n_pred)。預測順序 = 提交檔順序"""
    sum_r = sum_s = 0.0
    n_gt = n_pred = 0
    for img in set(gt_per_img) | set(pred_per_img):
        G = gt_per_img.get(img, [])
        P = pred_per_img.get(img, [])
        n_gt += len(G)
        n_pred += len(P)
        if not G or not P:
            continue
        ga = np.array([g.area for g in G])
        pa = np.array([p.area for p in P])
        inter = np.zeros((len(G), len(P)))
        for i, g in enumerate(G):
            if ga[i] <= 0:
                continue
            for j, p in enumerate(P):
                if pa[j] <= 0 or not g.intersects(p):
                    continue
                inter[i, j] = g.intersection(p).area

        # recall：每個 GT 找「交集面積最大」的預測（同分取較前面）
        for i in range(len(G)):
            j = int(np.argmax(inter[i]))
            if ga[i] <= 0 or inter[i, j] <= 0:
                continue
            iou = inter[i, j] / (ga[i] + pa[j] - inter[i, j])
            sum_r += iou * inter[i, j] / ga[i]

        # precision：每個預測找「交集面積最大」的 GT，並扣掉落在其他 GT 的部分
        s = np.zeros(len(P))
        owner = {}
        for j in range(len(P)):
            i = int(np.argmax(inter[:, j]))
            if pa[j] <= 0 or inter[i, j] <= 0:
                continue
            iou = inter[i, j] / (ga[i] + pa[j] - inter[i, j])
            others = [G[k] for k in range(len(G)) if k != i and inter[k, j] > 0]
            o = 0.0
            if others:
                o = P[j].intersection(unary_union(others).difference(G[i])).area / pa[j]
            s[j] = iou * (1.0 - o)
            owner[j] = i

        # 重複偵測：同一個 GT 只留 s 最高的（同分留較前面的）
        by_gt = {}
        for j, i in owner.items():
            by_gt.setdefault(i, []).append(j)
        for js in by_gt.values():
            keep = max(js, key=lambda j: (s[j], -j))
            for j in js:
                if j != keep:
                    s[j] = 0.0
        sum_s += float(s.sum())
    return sum_r, n_gt, sum_s, n_pred


def f1_tiou(gt, pred, classes=("solar_panel", "crack", "hotspot")):
    """gt / pred: {類別: {影像: [shapely 多邊形]}}。回傳 (F1_TIoU, {類別: F1_c})"""
    f1s = {}
    for c in classes:
        g, p = gt.get(c, {}), pred.get(c, {})
        if sum(map(len, g.values())) == 0 and sum(map(len, p.values())) == 0:
            continue  # 該類別既沒有 GT 也沒有預測，不列入平均
        sr, ng, sp, npred = class_scores(g, p)
        R = sr / ng if ng else 0.0
        P = sp / npred if npred else 0.0
        f1s[c] = 2 * R * P / (R + P) if R + P > 0 else 0.0
    return (float(np.mean(list(f1s.values()))) if f1s else 0.0), f1s


def match_pairs(G, P, thr=0.5):
    """README 第 6 節：IoU>=0.5、依 IoU 由大到小貪婪一對一配對。回傳 [(gt_idx, pred_idx)]"""
    cands = []
    for i, g in enumerate(G):
        for j, p in enumerate(P):
            if g.area <= 0 or p.area <= 0 or not g.intersects(p):
                continue
            inter = g.intersection(p).area
            iou = inter / (g.area + p.area - inter)
            if iou >= thr:
                cands.append((-iou, i, j))
    cands.sort()
    used_g, used_p, out = set(), set(), []
    for _, i, j in cands:
        if i in used_g or j in used_p:
            continue
        used_g.add(i)
        used_p.add(j)
        out.append((i, j))
    return out


# ---------- 驗證集 ground truth ----------

def load_gt(modality, split_name="val"):
    """回傳 (影像 id 清單, gt {類別: {key: [polygon]}}, reasons {key: [reason...]})。key = 'EO/train_0001'"""
    split_df = pd.read_csv(OUT / "train_val_split.csv")
    ids = split_df.loc[split_df[f"{modality}_split"] == split_name, "frame_id"].tolist()
    defect = CFG[modality]["defect"]
    gt = {"solar_panel": {}, defect: {}}
    reasons = {}
    for fid in ids:
        data = json.loads((TRAIN_ROOT / modality / "label" / f"{fid}.json").read_text(encoding="utf-8"))
        key = f"{modality}/{fid}"
        gt["solar_panel"].setdefault(key, [])
        gt[defect].setdefault(key, [])
        for s in data.get("shapes", []):
            lab = s.get("label")
            if lab not in gt:
                continue
            gt[lab][key].append(to_poly(s["points"]))
            if lab == defect:
                reasons.setdefault(key, []).append((s.get("reason") or "").strip())
    return ids, gt, reasons


# ---------- 批次預測 ----------

def predict_raw(model, paths, imgsz, conf=0.02, iou=0.6, max_det=300, batch=8):
    """對一串影像預測，回傳每張影像的預測 list（已依信心度由高到低排序）。
    每筆: dict(label, conf, box[x1,y1,x2,y2], points[N,2] 原圖座標, hw=(h,w))"""
    out = []
    for i in range(0, len(paths), batch):
        chunk = [p if isinstance(p, np.ndarray) else str(p) for p in paths[i:i + batch]]
        results = model.predict(source=chunk, imgsz=imgsz, conf=conf, iou=iou,
                                max_det=max_det, device=DEVICE, verbose=False, batch=batch)
        for r in results:
            h, w = r.orig_shape
            preds = []
            if r.boxes is not None and len(r.boxes) > 0:
                cls = r.boxes.cls.cpu().numpy().astype(int)
                cf = r.boxes.conf.cpu().numpy()
                xyxy = r.boxes.xyxy.cpu().numpy()
                polys = r.masks.xy if r.masks is not None else [None] * len(cls)
                for k in range(len(cls)):
                    pts = polys[k] if k < len(polys) else None
                    if pts is None or len(pts) < 3:
                        x1, y1, x2, y2 = xyxy[k].tolist()
                        pts = np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]], dtype=np.float32)
                    preds.append(dict(label=r.names[int(cls[k])], conf=float(cf[k]),
                                      box=xyxy[k].tolist(), points=np.asarray(pts, dtype=np.float32),
                                      hw=(h, w)))
            preds.sort(key=lambda d: -d["conf"])
            out.append(preds)
    return out


def resolve(preds, t_panel, t_anom, overlap=0.5):
    """套用門檻並處理同一模組的重複類別：異常優先，與異常框重疊的 solar_panel 框會被移除。
    回傳 (異常預測, 正常模組預測)，各自仍依信心度排序"""
    anom = [p for p in preds if p["label"] != "solar_panel" and p["conf"] >= t_anom]
    panel = [p for p in preds if p["label"] == "solar_panel" and p["conf"] >= t_panel
             and not any(box_iou(p["box"], a["box"]) >= overlap for a in anom)]
    return anom, panel

def load_source(path, modality):
    """偵測模型的輸入。IR 一律轉成灰階：訓練集 0% 是偽彩色，測試集約 25% 是偽彩色，
    不轉的話偽彩色那批幾乎找不到模組（框品質 0.16 -> 轉灰階後 0.83）。"""
    if modality == "IR":
        g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        return cv2.cvtColor(g, cv2.COLOR_GRAY2BGR)
    return str(path)


def colorfulness(img):
    """非黑像素中顏色鮮豔（通道差 > 40）的比例；> 0.3 視為偽彩色圖"""
    nb = img.max(axis=2) > 10
    if nb.sum() == 0:
        return 0.0
    d = (img.max(axis=2).astype(int) - img.min(axis=2).astype(int))[nb]
    return float((d > 40).mean())


print("V2-B 工具函式載入完成")


V2-B 工具函式載入完成


## 5. 驗證集評分（V2-C）：偵測模型單獨的表現與門檻

In [8]:
# ==========================================
# V2-C. 驗證集評分 + 門檻搜尋
# 用「只用 train split 訓練的模型」預測 val split，依官方公式算 F1_TIoU，
# 並找出每個模態最好的 solar_panel / 異常信心度門檻。
# ==========================================
import json
import time
import numpy as np
import pandas as pd
from ultralytics import YOLO

VAL_CONF = 0.02                     # 預測時用的最低信心度，之後再用門檻過濾
T_PANEL = [0.25, 0.35, 0.45, 0.55]  # solar_panel 門檻候選
T_ANOM = [0.05, 0.15, 0.30, 0.45]   # crack / hotspot 門檻候選


def get_poly(p, use_reg):
    key = "_poly_reg" if use_reg else "_poly_raw"
    if key not in p:
        h, w = p["hw"]
        pts = p["points"]
        if use_reg:
            reg = regularize(pts)
            pts = reg if reg is not None else pts
        p[key] = to_poly(clip_points(pts, w, h))
    return p[key]


def build_pred(raw, defect, t_panel, t_anom, use_reg):
    pred = {"solar_panel": {}, defect: {}}
    for key, preds in raw.items():
        anom, panel = resolve(preds, t_panel, t_anom)
        pred["solar_panel"][key] = [get_poly(p, use_reg) for p in panel]
        pred[defect][key] = [get_poly(p, use_reg) for p in anom]
    return pred


# ---------- 1. 對 val 影像預測一次（低信心度），結果存在記憶體 ----------
VAL = {}
for modality in ("EO", "IR"):
    ids, gt, reasons = load_gt(modality, "val")
    model = YOLO(str(val_weights(modality)))
    paths = [load_source(TRAIN_ROOT / modality / "image" / f"{fid}.jpg", modality) for fid in ids]
    t0 = time.time()
    raws = predict_raw(model, paths, CFG[modality]["imgsz"], conf=VAL_CONF)
    VAL[modality] = dict(
        raw={f"{modality}/{fid}": r for fid, r in zip(ids, raws)},
        gt=gt,
        reasons=reasons,
    )
    print(f"[{modality}] val 影像 {len(ids)} 張，預測用時 {(time.time() - t0) / 60:.1f} 分鐘")

# ---------- 2. 搜尋門檻 ----------
BEST = {}
for modality in ("EO", "IR"):
    defect = CFG[modality]["defect"]
    raw, gt = VAL[modality]["raw"], VAL[modality]["gt"]

    rows = []
    for t_anom in T_ANOM:
        for t_panel in T_PANEL:
            score, parts = f1_tiou(gt, build_pred(raw, defect, t_panel, t_anom, True))
            rows.append(dict(t_panel=t_panel, t_anom=t_anom, use_reg=True, score=score, **parts))
    best = max(rows, key=lambda r: r["score"])

    # 比較「不做凸包 / 頂點簡化」的版本
    score_raw, parts_raw = f1_tiou(gt, build_pred(raw, defect, best["t_panel"], best["t_anom"], False))
    if score_raw > best["score"]:
        best = dict(t_panel=best["t_panel"], t_anom=best["t_anom"], use_reg=False, score=score_raw, **parts_raw)
    BEST[modality] = best

    print(f"\n===== {modality}（{defect}）門檻搜尋，前 6 名 =====")
    display(pd.DataFrame(rows).sort_values("score", ascending=False).head(6).round(4))
    print(f"選用：t_panel={best['t_panel']}  t_anom={best['t_anom']}  use_reg={best['use_reg']}  "
          f"→ 該模態 F1_TIoU={best['score']:.4f}")

# ---------- 3. 兩個模態合併後的 F1_TIoU（與官方做法相同：solar_panel 是兩個模態合併計算）----------
pool_gt = {"solar_panel": {}, "crack": {}, "hotspot": {}}
pool_pred = {"solar_panel": {}, "crack": {}, "hotspot": {}}
for modality in ("EO", "IR"):
    b, defect = BEST[modality], CFG[modality]["defect"]
    pred = build_pred(VAL[modality]["raw"], defect, b["t_panel"], b["t_anom"], b["use_reg"])
    for c in pred:
        pool_pred[c].update(pred[c])
    for c in VAL[modality]["gt"]:
        pool_gt[c].update(VAL[modality]["gt"][c])
pooled, parts = f1_tiou(pool_gt, pool_pred)

print("\n" + "=" * 60)
print(f"驗證集 F1_TIoU = {pooled:.4f}")
for c, v in parts.items():
    print(f"  TIoU_{c:12s}= {v:.4f}")

# 異常框能否「配對成功」(IoU>=0.5)，決定 reason 會不會被計分
for modality in ("EO", "IR"):
    b, defect = BEST[modality], CFG[modality]["defect"]
    total = matched = 0
    for key, G in VAL[modality]["gt"][defect].items():
        anom, _ = resolve(VAL[modality]["raw"][key], b["t_panel"], b["t_anom"])
        P = [get_poly(p, b["use_reg"]) for p in anom]
        matched += len(match_pairs(G, P))
        total += len(G)
    print(f"  {defect:8s} GT 配對成功：{matched}/{total}（{matched / max(total, 1):.1%}）")

thresholds = {m: {k: (bool(v) if k == "use_reg" else float(v)) for k, v in BEST[m].items()
                  if k in ("t_panel", "t_anom", "use_reg", "score")} for m in BEST}
thresholds["pooled_f1_tiou"] = float(pooled)
(OUT / "thresholds.json").write_text(json.dumps(thresholds, ensure_ascii=False, indent=2), encoding="utf-8")
print("已存：", OUT / "thresholds.json")

sel_path = OUT / "reasons_selected.json"
if sel_path.exists():
    f1_text = json.loads(sel_path.read_text(encoding="utf-8"))["proxy_f1_text"]
    print(f"\n（估計）F1_TEXT≈{f1_text:.3f}  →  總分≈{0.7 * pooled + 0.3 * f1_text:.4f}（目標 0.8）")
else:
    print("\n接著執行 V2-D（挑選 reason）就能看到含文字分數的總分估計。")


[EO] val 影像 288 張，預測用時 0.5 分鐘
[IR] val 影像 305 張，預測用時 0.3 分鐘

===== EO（crack）門檻搜尋，前 6 名 =====


,t_panel,t_anom,use_reg,score,solar_panel,crack
15,0.55,0.45,True,0.4424,0.8848,0.0
11,0.55,0.30,True,0.4424,0.8848,0.0
7,0.55,0.15,True,0.4421,0.8842,0.0
3,0.55,0.05,True,0.4419,0.8837,0.0
10,0.45,0.30,True,0.4390,0.8780,0.0
14,0.45,0.45,True,0.4390,0.8780,0.0


選用：t_panel=0.55  t_anom=0.3  use_reg=False  → 該模態 F1_TIoU=0.4552

===== IR（hotspot）門檻搜尋，前 6 名 =====


,t_panel,t_anom,use_reg,score,solar_panel,hotspot
3,0.55,0.05,True,0.4615,0.9156,0.0074
2,0.45,0.05,True,0.4612,0.9151,0.0074
1,0.35,0.05,True,0.4605,0.9136,0.0074
0,0.25,0.05,True,0.4600,0.9125,0.0074
7,0.55,0.15,True,0.4581,0.9154,0.0007
11,0.55,0.30,True,0.4579,0.9154,0.0003


選用：t_panel=0.55  t_anom=0.05  use_reg=False  → 該模態 F1_TIoU=0.4689

驗證集 F1_TIoU = 0.3093
  TIoU_solar_panel = 0.9204
  TIoU_crack       = 0.0000
  TIoU_hotspot     = 0.0075
  crack    GT 配對成功：0/34（0.0%）
  hotspot  GT 配對成功：1/216（0.5%）
已存： C:\Users\Sharon\Downloads\VLPaicup\outputs\thresholds.json

（估計）F1_TEXT≈0.893  →  總分≈0.4843（目標 0.8）


## 5b. 挑選 reason 文字（BERTScore）

In [9]:
# ==========================================
# V2-D. 挑選 reason 文字（F1_TEXT 佔總分 30%）
# 官方用 BERTScore（bert-base-chinese、第 8 層、rescale）把 reason 跟「該成因類別的參考描述」比，取最高分。
# 參考描述不公開，但訓練標註的 reason 同樣是這類句子，所以：
#   1. 把訓練 reason 依語意相似度分群（當作成因類別的替身）
#   2. 找出一句「對每個群都夠像」的 reason（各群分數取調和平均，跟官方彙整方式一致）
# 需要：pip install bert-score transformers torch（第一次會下載 bert-base-chinese）
# ==========================================
import json
import itertools
import numpy as np
from collections import Counter
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from bert_score import BERTScorer

K_LIST = (2, 3, 4, 5)   # 不知道真正有幾個成因類別，所以對多種分群數都求平均，挑最穩的一句

scorer = BERTScorer(model_type="bert-base-chinese", num_layers=8, lang="zh", rescale_with_baseline=True)


def pair_scores(cands, refs):
    """逐對計算 rescale 後的 BERTScore F1（cands、refs 等長）"""
    _, _, F = scorer.score(list(cands), list(refs), batch_size=64)
    return F.cpu().numpy()


def score_matrix(cands, refs):
    n, m = len(cands), len(refs)
    c = [a for a in cands for _ in refs]
    r = [b for _ in cands for b in refs]
    return pair_scores(c, r).reshape(n, m)


def load_reason_counts(modality, defect):
    counts = Counter()
    for p in sorted((TRAIN_ROOT / modality / "label").glob("*.json")):
        data = json.loads(p.read_text(encoding="utf-8"))
        for s in data.get("shapes", []):
            if s.get("label") == defect:
                r = (s.get("reason") or "").strip()
                if r:
                    counts[r] += 1
    return counts


def cluster_labels(S, k):
    D = 1.0 - (S + S.T) / 2.0
    np.fill_diagonal(D, 0.0)
    D = np.clip(D, 0.0, None)
    Z = linkage(squareform(D, checks=False), method="average")
    return fcluster(Z, t=k, criterion="maxclust")


def hmean(values):
    values = list(values)
    if not values or any(v <= 0 for v in values):
        return 0.0
    return len(values) / sum(1.0 / v for v in values)


def cluster_max(Sc, labels):
    """每個候選對每個群的分數 = 與該群所有參考句分數的最大值（官方也是取最高）。回傳 (n_cand, n_cluster)"""
    return np.stack([Sc[:, labels == c].max(axis=1) for c in sorted(set(labels))], axis=1)


def avg_over_k(Sc, labels_by_k):
    return np.mean([[hmean(row) for row in cluster_max(Sc, lab)] for lab in labels_by_k.values()], axis=0)


SEL = {}
cluster_vectors = {}   # modality -> {k: 該群分數向量}，最後合併算 F1_TEXT 估計
for modality, defect in (("EO", "crack"), ("IR", "hotspot")):
    counts = load_reason_counts(modality, defect)
    U = list(counts)
    print(f"\n===== {modality}/{defect}：{len(U)} 種不同的 reason，共 {sum(counts.values())} 筆 =====")

    S = score_matrix(U, U)
    labels_by_k = {k: cluster_labels(S, k) for k in K_LIST if k <= len(U)}

    # 候選 = 每一句訓練 reason + 分數最高幾句兩兩串接（串接後仍須 <= 500 字）
    base_avg = avg_over_k(S, labels_by_k)
    top = list(np.argsort(-base_avg)[:8])
    extra = []
    for a, b in itertools.combinations(top, 2):
        text = U[a].rstrip("。") + "，" + U[b]
        if len(text) <= 500:
            extra.append(text)
    cands = U + extra
    Sc = np.vstack([S, score_matrix(extra, U)]) if extra else S
    avg = avg_over_k(Sc, labels_by_k)

    best_i = int(np.argmax(avg))
    best_text = cands[best_i]
    per_k = {k: float(np.mean([hmean(row) for row in cluster_max(Sc[[best_i]], lab)])) for k, lab in labels_by_k.items()}
    cluster_vectors[modality] = {k: cluster_max(Sc[[best_i]], lab)[0].tolist() for k, lab in labels_by_k.items()}

    print("分群結果（k=3）：")
    lab3 = labels_by_k.get(3, next(iter(labels_by_k.values())))
    for c in sorted(set(lab3)):
        members = [u for u, l in zip(U, lab3) if l == c]
        print(f"  群 {c}：{len(members)} 句，例：{members[0]}")
    print("選中的 reason：", best_text)
    print("各分群數下的調和平均分數：", {k: round(v, 3) for k, v in per_k.items()})
    SEL[modality] = dict(reason=best_text, mean_hm=float(avg[best_i]), per_k=per_k)

# 官方 raw 是「所有成因類別（crack + hotspot）」一起取調和平均，這裡用同樣方式估計
raw_by_k = []
for k in K_LIST:
    vals = []
    for modality in ("EO", "IR"):
        vals += cluster_vectors[modality].get(k, [])
    raw_by_k.append(hmean(vals))
proxy_raw = float(np.mean(raw_by_k))
proxy_f1_text = float(min(max((proxy_raw - 0.30) / 0.40, 0.0), 1.0))
print(f"\n估計 raw={proxy_raw:.3f} → F1_TEXT≈{proxy_f1_text:.3f}（這是用訓練 reason 當替身的估計，不等於排行榜分數）")

result = dict(crack=dict(reason=SEL["EO"]["reason"]), hotspot=dict(reason=SEL["IR"]["reason"]),
              proxy_raw=proxy_raw, proxy_f1_text=proxy_f1_text, details=SEL)
(OUT / "reasons_selected.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print("已存：", OUT / "reasons_selected.json")

th_path = OUT / "thresholds.json"
if th_path.exists():
    pooled = json.loads(th_path.read_text(encoding="utf-8"))["pooled_f1_tiou"]
    print(f"（估計）驗證集 F1_TIoU={pooled:.4f}、F1_TEXT≈{proxy_f1_text:.3f}  →  總分≈{0.7 * pooled + 0.3 * proxy_f1_text:.4f}（目標 0.8）")


c:\Users\Sharon\Downloads\VLPaicup\.venv-xpu214\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1464.40it/s]
[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when


===== EO/crack：49 種不同的 reason，共 352 筆 =====


c:\Users\Sharon\Downloads\VLPaicup\.venv-xpu214\Lib\site-packages\bert_score\scorer.py:139: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  self._baseline_vals = torch.from_numpy(


分群結果（k=3）：
  群 1：5 句，例：模組表面出現明顯裂紋，推測為外部物理衝擊所造成。
  群 2：27 句，例：面板出現裂損，導電路徑於該處受阻。
  群 3：17 句，例：該片模組的板體破裂，損傷不僅止於表面。
選中的 reason： 模組面板有局部實體損傷，表面已破損。
各分群數下的調和平均分數： {2: 0.698, 3: 0.684, 4: 0.643, 5: 0.623}

===== IR/hotspot：80 種不同的 reason，共 444 筆 =====
分群結果（k=3）：
  群 1：17 句，例：該區呈條狀高溫，疑為旁路二極體導通或部分電池串失效所致。
  群 2：24 句，例：可能為內部隱裂、接觸電阻上升或電性異常，進而引發局部過熱。
  群 3：39 句，例：模組板破裂造成電流分布不均，局部電阻上升並形成異常發熱區。
選中的 reason： 推測模組內部存在隱裂或電性異常，導致該區域電阻偏高而發熱。
各分群數下的調和平均分數： {2: 0.704, 3: 0.664, 4: 0.636, 5: 0.606}

估計 raw=0.657 → F1_TEXT≈0.893（這是用訓練 reason 當替身的估計，不等於排行榜分數）
已存： C:\Users\Sharon\Downloads\VLPaicup\outputs\reasons_selected.json
（估計）驗證集 F1_TIoU=0.3093、F1_TEXT≈0.893  →  總分≈0.4843（目標 0.8）


## 5c.（選用）偵測模型全資料重訓

In [10]:
# ==========================================
# 5.（選用，預設關閉）用全部訓練資料重訓偵測模型，EO 約 7 小時、IR 約 5 小時。
# 驗證集那一組（EO 是 G01、IR 是 G04）占了 27~29% 的資料，加回去通常會更準。
# 門檻沿用 V2-C 在 val 上調出來的結果。不想花時間可以跳過，推論會自動改用 V2 的 best.pt。
# ==========================================
RUN_FINAL_FIT = False

if RUN_FINAL_FIT:
    train_one("EO", final=True)
    train_one("IR", final=True)

print("EO 推論用權重：", best_weights("EO"))
print("IR 推論用權重：", best_weights("IR"))


EO 推論用權重： C:\Users\Sharon\Downloads\VLPaicup\outputs\runs\eo_v2\weights\best.pt
IR 推論用權重： C:\Users\Sharon\Downloads\VLPaicup\outputs\runs\ir_v2\weights\best.pt


## 6. 模組相對特徵 + 整組留出驗證（hotspot 的主力）

In [11]:
# ==========================================
# 6. 以「模組」為單位的相對特徵（IR hotspot 的主力）
#    每個模組的亮度/邊緣統計，並與同一張圖的其他模組比較（差值與 z 分數）。
#    這是物理特徵（比鄰居更熱），比「記得某個場景」更能跨場景。
# ==========================================
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

STATS = ["mean", "std", "p95", "vmax", "gmean", "gp95", "lmean"]
BASE = ["mean", "std", "p05", "p95", "vmax", "gmean", "gp95", "lmean", "area"]
THR = [0.01, 0.02, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]


def image_feats(gray, polys, scale):
    """gray: 原圖灰階；polys: 原圖座標的多邊形清單。回傳每個模組一列的特徵表"""
    g = gray if scale == 1 else cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    gm = np.hypot(cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1))
    lap = np.abs(cv2.Laplacian(g, cv2.CV_32F))
    rows = []
    for pts0 in polys:
        pts = (np.asarray(pts0) * scale).astype(np.int32)
        m = np.zeros(g.shape, np.uint8)
        cv2.fillPoly(m, [pts], 1)
        m = cv2.erode(m, np.ones((5, 5), np.uint8))
        if m.sum() < 50:
            rows.append({k: np.nan for k in BASE})
            continue
        v, gg, ll = g[m > 0].astype(np.float32), gm[m > 0], lap[m > 0]
        rows.append(dict(mean=v.mean(), std=v.std(), p05=np.percentile(v, 5), p95=np.percentile(v, 95),
                         vmax=v.max(), gmean=gg.mean(), gp95=np.percentile(gg, 95), lmean=ll.mean(),
                         area=float(m.sum())))
    df = pd.DataFrame(rows, columns=BASE)
    for c in STATS:
        med, sd = df[c].median(), df[c].std() + 1e-6
        df[c + "_d"] = df[c] - med
        df[c + "_z"] = (df[c] - med) / sd
    return df


def fit_clf(F, feats):
    w = np.where(F.y == 1, (F.y == 0).sum() / max(F.y.sum(), 1), 1.0)
    clf = HistGradientBoostingClassifier(max_depth=4, max_iter=200, learning_rate=0.05)
    return clf.fit(F[feats], F.y, sample_weight=w)


def clean_modules(preds, t_panel):
    """不分類別：依信心度做 NMS，並去掉相對過小的碎框"""
    cand = sorted([p for p in preds if p["conf"] >= t_panel], key=lambda p: -p["conf"])
    keep = []
    for p in cand:
        if all(box_iou(p["box"], q["box"]) < 0.5 for q in keep):
            keep.append(p)
    if len(keep) >= 4:
        areas = np.array([(p["box"][2] - p["box"][0]) * (p["box"][3] - p["box"][1]) for p in keep])
        ref = np.percentile(areas, 75)
        keep = [p for p, a in zip(keep, areas) if a >= 0.2 * ref]
    return keep


def module_feats(modality):
    split = pd.read_csv(OUT / "train_val_split.csv")
    grp = dict(zip(split.frame_id, split.group))
    defect = CFG[modality]["defect"]
    tables = []
    for n, fid in enumerate(split.frame_id):
        gray = cv2.imread(str(TRAIN_ROOT / modality / "image" / f"{fid}.jpg"), cv2.IMREAD_GRAYSCALE)
        d = json.loads((TRAIN_ROOT / modality / "label" / f"{fid}.json").read_text(encoding="utf-8"))
        shapes = [s for s in d["shapes"] if s["label"] in ("solar_panel", defect)]
        if not shapes:
            continue
        df = image_feats(gray, [s["points"] for s in shapes], SCALE[modality])
        df.insert(0, "frame", fid)
        df.insert(1, "group", grp[fid])
        df.insert(2, "y", [int(s["label"] == defect) for s in shapes])
        tables.append(df.dropna(subset=["mean"]))
        if n % 200 == 0:
            print(f"  [{modality}] {n}/{len(split)}")
    return pd.concat(tables, ignore_index=True)


MODFEAT = {}
for modality in ("IR", "EO"):
    path = OUT / f"module_feats_{modality}.csv"
    if path.exists():
        F = pd.read_csv(path)
        print(f"[{modality}] 讀取已存的模組特徵：{path.name}")
    else:
        F = module_feats(modality)
        F.to_csv(path, index=False)
    MODFEAT[modality] = F
    feats = [c for c in F.columns if c not in ("frame", "group", "y")]
    print(f"\n===== {modality}：模組 {len(F)} 個，異常 {int(F.y.sum())} 個（整組留出驗證）=====")
    for g in sorted(F.group[F.y == 1].unique()):
        te = F.group == g
        if F.y[te].sum() < 5:
            continue
        clf = fit_clf(F[~te], feats)
        p = clf.predict_proba(F.loc[te, feats])[:, 1]
        print(f"  留出 {g}：異常 {int(F.y[te].sum())}/{int(te.sum())}  AUC={roc_auc_score(F.y[te], p):.3f}  "
              f"AP={average_precision_score(F.y[te], p):.3f}  (隨機 AP≈{F.y[te].mean():.3f})")


[IR] 讀取已存的模組特徵：module_feats_IR.csv

===== IR：模組 26697 個，異常 444 個（整組留出驗證）=====
  留出 G01：異常 7/2074  AUC=0.987  AP=0.170  (隨機 AP≈0.003)
  留出 G02：異常 190/15860  AUC=0.969  AP=0.777  (隨機 AP≈0.012)
  留出 G03：異常 31/487  AUC=0.955  AP=0.702  (隨機 AP≈0.064)
  留出 G04：異常 216/2581  AUC=0.994  AP=0.981  (隨機 AP≈0.084)
[EO] 讀取已存的模組特徵：module_feats_EO.csv

===== EO：模組 29905 個，異常 352 個（整組留出驗證）=====
  留出 G01：異常 34/2371  AUC=0.824  AP=0.053  (隨機 AP≈0.014)
  留出 G04：異常 310/2368  AUC=0.482  AP=0.196  (隨機 AP≈0.131)
  留出 G07：異常 8/301  AUC=0.913  AP=0.149  (隨機 AP≈0.027)


## 7. EO crack：ResNet 特徵工具

In [12]:
# ==========================================
# 7. EO crack 用：預訓練 ResNet18 對每個模組裁切小圖取特徵
#    （第一次執行會下載 ResNet18 權重）
# ==========================================
import torch
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

dev = torch.device("xpu" if torch.xpu.is_available() else "cpu")
net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
net.fc = torch.nn.Identity()
net = net.eval().to(dev)
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)


def crops_from(img, polys, size=224):
    """每個多邊形裁出外接矩形，多邊形以外塗黑，縮成 size x size"""
    out = []
    H, W = img.shape[:2]
    for pts in polys:
        pts = np.asarray(pts, np.float32)
        x0, y0 = np.floor(pts.min(0)).astype(int)
        x1, y1 = np.ceil(pts.max(0)).astype(int)
        x0, y0, x1, y1 = max(x0, 0), max(y0, 0), min(x1, W), min(y1, H)
        if x1 - x0 < 8 or y1 - y0 < 8:
            out.append(np.zeros((size, size, 3), np.uint8))
            continue
        m = np.zeros((y1 - y0, x1 - x0), np.uint8)
        cv2.fillPoly(m, [(pts - [x0, y0]).astype(np.int32)], 1)
        c = img[y0:y1, x0:x1] * m[:, :, None]
        out.append(cv2.resize(c, (size, size), interpolation=cv2.INTER_AREA))
    return out


@torch.no_grad()
def embed(crops, bs=64):
    res = []
    for i in range(0, len(crops), bs):
        x = torch.from_numpy(np.stack(crops[i:i + bs])[..., ::-1].copy()).to(dev).permute(0, 3, 1, 2).float() / 255
        res.append(net((x - MEAN) / STD).float().cpu().numpy())
    return np.concatenate(res)


def module_embed(img, polys):
    """回傳 [相對於同一張圖平均的特徵, 原始特徵]"""
    e = embed(crops_from(img, polys))
    return np.concatenate([e - e.mean(0, keepdims=True), e], axis=1)


print("裝置：", dev)


裝置： xpu


## 8. EO crack：取特徵 + 整組留出驗證

In [13]:
# 8. 對所有訓練 EO 模組取特徵（已存過就讀檔），並做整組留出驗證
EMB_PATH = OUT / "eo_embed.npz"
if EMB_PATH.exists():
    Z = np.load(EMB_PATH, allow_pickle=True)
    E_eo, M_eo = Z["E"], pd.DataFrame(dict(frame=Z["frame"], group=Z["group"], y=Z["y"]))
    print("讀取已存的 EO 特徵：", EMB_PATH.name, E_eo.shape)
else:
    split = pd.read_csv(OUT / "train_val_split.csv")
    grp = dict(zip(split.frame_id, split.group))
    E, meta = [], []
    for n, fid in enumerate(split.frame_id):
        img = cv2.imread(str(TRAIN_ROOT / "EO" / "image" / f"{fid}.jpg"))
        d = json.loads((TRAIN_ROOT / "EO" / "label" / f"{fid}.json").read_text(encoding="utf-8"))
        shapes = [s for s in d["shapes"] if s["label"] in ("solar_panel", "crack")]
        if not shapes:
            continue
        E.append(module_embed(img, [s["points"] for s in shapes]))
        meta += [(fid, grp[fid], int(s["label"] == "crack")) for s in shapes]
        if n % 100 == 0:
            print(f"{n}/{len(split)}")
    E_eo = np.concatenate(E)
    M_eo = pd.DataFrame(meta, columns=["frame", "group", "y"])
    np.savez_compressed(EMB_PATH, E=E_eo, frame=M_eo.frame.values, group=M_eo.group.values, y=M_eo.y.values)

print(f"\n模組 {len(M_eo)}，crack {int(M_eo.y.sum())}（整組留出驗證）")
for g in sorted(M_eo.group[M_eo.y == 1].unique()):
    te = (M_eo.group == g).values
    if M_eo.y[te].sum() < 5:
        continue
    sc = StandardScaler().fit(E_eo[~te])
    lr = LogisticRegression(C=0.01, class_weight="balanced", max_iter=2000).fit(sc.transform(E_eo[~te]), M_eo.y[~te])
    p = lr.predict_proba(sc.transform(E_eo[te]))[:, 1]
    print(f"  留出 {g}：crack {int(M_eo.y[te].sum())}/{int(te.sum())}  AUC={roc_auc_score(M_eo.y[te], p):.3f}  "
          f"AP={average_precision_score(M_eo.y[te], p):.3f}  (隨機 AP≈{M_eo.y[te].mean():.3f})")


讀取已存的 EO 特徵： eo_embed.npz (29905, 1024)

模組 29905，crack 352（整組留出驗證）
  留出 G01：crack 34/2371  AUC=0.625  AP=0.018  (隨機 AP≈0.014)
  留出 G04：crack 310/2368  AUC=0.734  AP=0.558  (隨機 AP≈0.131)
  留出 G07：crack 8/301  AUC=0.994  AP=0.807  (隨機 AP≈0.027)


## 9. 兩階段端到端驗證 + 儲存分類器

In [14]:
# ==========================================
# 9. 兩階段端到端驗證（用「真正的預測框」，不是標註框）
#    偵測模型與分類器都沒看過驗證組（IR=G04、EO=G01）。依官方公式算 F1_TIoU，並掃描分類門檻。
#    需要先執行過 V2-C（驗證集評分）那一格，因為要用到 VAL。
# ==========================================
import joblib

split = pd.read_csv(OUT / "train_val_split.csv")
TWO = {}
for modality in ("IR", "EO"):
    defect = CFG[modality]["defect"]
    F = MODFEAT[modality]
    feats = [c for c in F.columns if c not in ("frame", "group", "y")]
    hold = split.loc[split[f"{modality}_split"] == "val", "group"].iloc[0]
    clf = fit_clf(F[F.group != hold], feats)

    raw, gt = VAL[modality]["raw"], VAL[modality]["gt"]
    mods, probs = {}, {}
    for key, preds in raw.items():
        ms = clean_modules(preds, T_PANEL_V3)
        if not ms:
            continue
        fid = key.split("/")[1]
        gray = cv2.imread(str(TRAIN_ROOT / modality / "image" / f"{fid}.jpg"), cv2.IMREAD_GRAYSCALE)
        X = image_feats(gray, [p["points"] for p in ms], SCALE[modality])
        mods[key] = ms
        probs[key] = clf.predict_proba(X[feats])[:, 1]

    rows = []
    for thr in THR:
        pred = {"solar_panel": {}, defect: {}}
        for key, ms in mods.items():
            polys = [get_poly(p, True) for p in ms]
            flag = probs[key] >= thr
            pred[defect][key] = [q for q, f in zip(polys, flag) if f]
            pred["solar_panel"][key] = [q for q, f in zip(polys, flag) if not f]
        score, parts = f1_tiou(gt, pred)
        matched = sum(len(match_pairs(gt[defect].get(k, []), pred[defect].get(k, []))) for k in gt[defect])
        rows.append(dict(thr=thr, score=score, matched=f"{matched}/{sum(map(len, gt[defect].values()))}", **parts))
    tab = pd.DataFrame(rows)
    print(f"\n===== {modality}（驗證組 {hold}）=====")
    print(tab.round(4).to_string(index=False))
    if modality == "IR":
        # 排名做法：每張圖取前 k 名且分數 >= tau（見設定格 HOT_K / HOT_TAU 的說明）
        rank_rows = []
        for k in (1, 2):
            for tau in (0, 1e-4, 1e-3, 1e-2):
                pred = {"solar_panel": {}, defect: {}}
                for key, ms in mods.items():
                    pr = probs[key]
                    flag = np.zeros(len(ms), bool)
                    flag[np.argsort(-pr)[:k]] = True
                    flag &= pr >= tau
                    polys = [get_poly(p, True) for p in ms]
                    pred[defect][key] = [q for q, f in zip(polys, flag) if f]
                    pred["solar_panel"][key] = [q for q, f in zip(polys, flag) if not f]
                score, parts = f1_tiou(gt, pred)
                matched = sum(len(match_pairs(gt[defect].get(k_, []), pred[defect].get(k_, []))) for k_ in gt[defect])
                rank_rows.append(dict(k=k, tau=tau, score=score, matched=f"{matched}/{sum(map(len, gt[defect].values()))}", **parts))
        print(f"\n----- {modality}：每張圖取前 k 名 + 地板值 tau（驗證組 {hold}）-----")
        print(pd.DataFrame(rank_rows).round(4).to_string(index=False))
    best = tab.loc[tab[defect].idxmax()]
    TWO[modality] = dict(thr=float(best.thr), t_panel=T_PANEL_V3, val_defect_f1=float(best[defect]))
    joblib.dump(dict(clf=fit_clf(F, feats), feats=feats), OUT / f"clf_{modality}.joblib")  # 全資料版，推論用

(OUT / "two_stage.json").write_text(json.dumps(TWO, ensure_ascii=False, indent=2), encoding="utf-8")
print("\n已存：", OUT / "two_stage.json", "與 clf_IR / clf_EO.joblib")



===== IR（驗證組 G04）=====
 thr  score matched  solar_panel  hotspot
0.01 0.8847 162/216       0.9470   0.8224
0.02 0.8585 146/216       0.9438   0.7732
0.03 0.8354 133/216       0.9413   0.7294
0.05 0.8108 120/216       0.9386   0.6829
0.10 0.7819 106/216       0.9359   0.6279
0.15 0.7626  97/216       0.9342   0.5911
0.20 0.7403  87/216       0.9322   0.5483
0.30 0.6773  62/216       0.9277   0.4269
0.40 0.6176  42/216       0.9239   0.3113
0.50 0.5450  21/216       0.9200   0.1699
0.60 0.4886   7/216       0.9174   0.0598
0.70 0.4757   4/216       0.9168   0.0346
0.80 0.4580   0/216       0.9161   0.0000
0.90 0.4580   0/216       0.9161   0.0000

----- IR：每張圖取前 k 名 + 地板值 tau（驗證組 G04）-----
 k    tau  score matched  solar_panel  hotspot
 1 0.0000 0.8597 211/216       0.9377   0.7817
 1 0.0001 0.9304 210/216       0.9531   0.9078
 1 0.0010 0.9203 188/216       0.9521   0.8885
 1 0.0100 0.8847 162/216       0.9470   0.8224
 2 0.0000 0.6892 216/216       0.8717   0.5066
 2 0.0001 0.9220 212

## 10. 框品質檢查（不需要答案，測試集 vs 驗證集）

In [15]:
# ==========================================
# 10. 不需要答案的「框品質」檢查
#     資料說明：標註框以外一律填黑（測試圖也是）。所以「非黑區域」≈ 標準答案的模組範圍。
#     比對預測框聯集與非黑區域，驗證圖（已知 F1≈0.9）當基準，測試圖拿來和它比。
#     上傳前，測試圖的數字應該要接近驗證圖，否則 solar_panel / hotspot 都不會有分。
# ==========================================
random.seed(3)


def cover_stats(img, ms):
    h, w = img.shape[:2]
    nb = (img.max(axis=2) > 10).astype(np.uint8)
    nb = cv2.morphologyEx(nb, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    pm = np.zeros((h, w), np.uint8)
    for p in ms:
        pts = final_points(p["points"], True, w, h)
        if pts is not None:
            cv2.fillPoly(pm, [np.round(np.array(pts)).astype(np.int32)], 1)
    inter = int((nb & pm).sum())
    union = int((nb | pm).sum())
    return inter / max(union, 1), 1 - inter / max(int(pm.sum()), 1), 1 - inter / max(int(nb.sum()), 1), nb, pm


all_files = pd.read_csv(KIT_ROOT / "test_images.csv")["filename"].tolist()
COVER = {}
for modality in ("EO", "IR"):
    vstats = []
    for key, preds in list(VAL[modality]["raw"].items())[:80]:
        img = cv2.imread(str(TRAIN_ROOT / modality / "image" / f"{key.split('/')[1]}.jpg"))
        vstats.append(cover_stats(img, clean_modules(preds, T_PANEL_V3))[:3])
    v = np.mean(vstats, axis=0)

    files = random.sample([f for f in all_files if f"/{modality}/" in f], 100)
    model = YOLO(str(best_weights(modality)))
    raws = predict_raw(model, [load_source(TEST_BASE / f, modality) for f in files], INFER_IMGSZ[modality],
                       conf=T_PANEL_V3, max_det=300, batch=INFER_BATCH[modality])
    tstats, items = [], []
    for f, raw in zip(files, raws):
        img = cv2.imread(str(TEST_BASE / f))
        ms = clean_modules(raw, T_PANEL_V3)
        iou, ob, mi, nb, pm = cover_stats(img, ms)
        tstats.append((iou, ob, mi))
        items.append((iou, f, img, nb, pm, len(ms)))
    t = np.mean(tstats, axis=0)
    if modality == "IR":
        col = np.array([colorfulness(it[2]) > 0.3 for it in items])
        ious = np.array([it[0] for it in items])
        print(f"   [IR] 偽彩色圖 {col.mean():.0%}：IoU 偽彩色 {ious[col].mean() if col.any() else float('nan'):.3f} / "
              f"非偽彩色 {ious[~col].mean() if (~col).any() else float('nan'):.3f}")
    COVER[modality] = dict(val_iou=float(v[0]), test_iou=float(t[0]))
    print(f"\n[{modality}] 預測框 vs 非黑區域：IoU / 框落在黑底比例 / 非黑區域漏框比例")
    print(f"   驗證集：{v[0]:.3f} / {v[1]:.3f} / {v[2]:.3f}")
    print(f"   測試集：{t[0]:.3f} / {t[1]:.3f} / {t[2]:.3f}")
    print(f"   測試集每張平均預測模組數：{np.mean([it[5] for it in items]):.1f}")

    tiles = []
    for iou, f, img, nb, pm, n in sorted(items, key=lambda x: x[0])[:6]:
        vis = img.copy()
        miss = (nb == 1) & (pm == 0)
        vis[miss] = (0.5 * vis[miss] + 0.5 * np.array([0, 0, 255])).astype(np.uint8)
        cnts, _ = cv2.findContours(pm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(vis, cnts, -1, (0, 255, 0), max(2, img.shape[1] // 500))
        s = 640 / img.shape[1]
        vis = cv2.resize(vis, (640, int(img.shape[0] * s)))
        cv2.putText(vis, f"{Path(f).stem} IoU={iou:.2f} n={n}", (6, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        tiles.append(vis)
    hmax = max(t_.shape[0] for t_ in tiles)
    tiles = [cv2.copyMakeBorder(t_, 0, hmax - t_.shape[0], 0, 0, cv2.BORDER_CONSTANT) for t_ in tiles]
    cv2.imwrite(str(OUT / f"diag_cover_{modality}.jpg"), np.vstack([np.hstack(tiles[:3]), np.hstack(tiles[3:])]))
    print("   最差 6 張疊圖（紅=沒框到的非黑區域，綠=預測框）：", OUT / f"diag_cover_{modality}.jpg")



[EO] 預測框 vs 非黑區域：IoU / 框落在黑底比例 / 非黑區域漏框比例
   驗證集：0.964 / 0.008 / 0.028
   測試集：0.947 / 0.003 / 0.049
   測試集每張平均預測模組數：58.5
   最差 6 張疊圖（紅=沒框到的非黑區域，綠=預測框）： C:\Users\Sharon\Downloads\VLPaicup\outputs\diag_cover_EO.jpg
   [IR] 偽彩色圖 31%：IoU 偽彩色 0.832 / 非偽彩色 0.963

[IR] 預測框 vs 非黑區域：IoU / 框落在黑底比例 / 非黑區域漏框比例
   驗證集：0.972 / 0.006 / 0.022
   測試集：0.922 / 0.034 / 0.049
   測試集每張平均預測模組數：49.4
   最差 6 張疊圖（紅=沒框到的非黑區域，綠=預測框）： C:\Users\Sharon\Downloads\VLPaicup\outputs\diag_cover_IR.jpg


## 11. 產生提交檔並做官方格式檢查

In [16]:
# ==========================================
# 11. 產生提交檔 my_submission_v5.json（IR：兩階段 hotspot；EO：模組 + 可選 crack）
#     上傳前請先看上一格的「框品質」：測試集 IoU 要接近驗證集，才值得上傳。
# ==========================================
TWO = json.loads((OUT / "two_stage.json").read_text(encoding="utf-8"))
print(f"hotspot：每張圖取前 {HOT_K} 名且分數 >= {HOT_TAU} | EO crack：{USE_EO_CRACK}（前 {CRACK_FRAC:.0%}）")

sel = json.loads((OUT / "reasons_selected.json").read_text(encoding="utf-8"))
REASON_HOT, REASON_CRACK = sel["hotspot"]["reason"], sel["crack"]["reason"]
clf_ir = joblib.load(OUT / "clf_IR.joblib")
all_files = pd.read_csv(KIT_ROOT / "test_images.csv")["filename"].tolist()
assert len(all_files) == 1392 and len(set(all_files)) == 1392
sub = {}

# ---- IR ----
ir_files = [f for f in all_files if "/IR/" in f]
model = YOLO(str(best_weights("IR")))
n_hot = n_ir_panel = n_hot_col = n_mod_col = 0
for i in range(0, len(ir_files), 8):
    chunk = ir_files[i:i + 8]
    raws = predict_raw(model, [load_source(TEST_BASE / f, "IR") for f in chunk], INFER_IMGSZ["IR"],
                       conf=T_PANEL_V3, max_det=300, batch=INFER_BATCH["IR"])
    for f, raw in zip(chunk, raws):
        ms = clean_modules(raw, T_PANEL_V3)
        flags = [False] * len(ms)
        is_col = colorfulness(cv2.imread(str(TEST_BASE / f))) > 0.3
        if ms:
            gray = cv2.imread(str(TEST_BASE / f), cv2.IMREAD_GRAYSCALE)
            X = image_feats(gray, [p["points"] for p in ms], SCALE["IR"])
            pr = clf_ir["clf"].predict_proba(X[clf_ir["feats"]])[:, 1]
            fl = np.zeros(len(ms), bool)
            fl[np.argsort(-pr)[:HOT_K]] = True
            flags = list(fl & (pr >= HOT_TAU))
        out = []
        for p, is_hot in zip(ms, flags):
            h, w = p["hw"]
            pts = final_points(p["points"], True, w, h)
            if pts is None:
                continue
            if is_hot:
                out.append({"label": "hotspot", "points": pts, "reason": REASON_HOT})
                n_hot += 1
                n_hot_col += int(is_col)
            else:
                out.append({"label": "solar_panel", "points": pts})
                n_ir_panel += 1
            n_mod_col += int(is_col)
        sub[f] = out
    print(f"[IR] {min(i + 8, len(ir_files))}/{len(ir_files)}")

# ---- EO ----
eo_files = [f for f in all_files if "/EO/" in f]
if USE_EO_CRACK:
    sc_all = StandardScaler().fit(E_eo)
    lr_all = LogisticRegression(C=0.01, class_weight="balanced", max_iter=2000).fit(sc_all.transform(E_eo), M_eo.y)
model = YOLO(str(best_weights("EO")))
store = []
for i in range(0, len(eo_files), 8):
    chunk = eo_files[i:i + 8]
    raws = predict_raw(model, [TEST_BASE / f for f in chunk], INFER_IMGSZ["EO"],
                       conf=T_PANEL_V3, max_det=300, batch=INFER_BATCH["EO"])
    for f, raw in zip(chunk, raws):
        ms = clean_modules(raw, T_PANEL_V3)
        pr = np.zeros(len(ms))
        if USE_EO_CRACK and ms:
            img = cv2.imread(str(TEST_BASE / f))
            pr = lr_all.predict_proba(sc_all.transform(module_embed(img, [p["points"] for p in ms])))[:, 1]
        store.append((f, ms, pr))
    print(f"[EO] {min(i + 8, len(eo_files))}/{len(eo_files)}")

allp = np.concatenate([s[2] for s in store])
thr_crack = float(np.quantile(allp, 1 - CRACK_FRAC)) if USE_EO_CRACK and len(allp) else 2.0
n_crack = n_eo_panel = 0
for f, ms, pr in store:
    out = []
    for m, p_ in zip(ms, pr):
        h, w = m["hw"]
        pts = final_points(m["points"], True, w, h)
        if pts is None:
            continue
        if USE_EO_CRACK and p_ >= thr_crack:
            out.append({"label": "crack", "points": pts, "reason": REASON_CRACK})
            n_crack += 1
        else:
            out.append({"label": "solar_panel", "points": pts})
            n_eo_panel += 1
    sub[f] = out

assert len(sub) == len(all_files)
V5_PATH = OUT / "my_submission_v5.json"
V5_PATH.write_text(json.dumps({f: sub[f] for f in all_files}, ensure_ascii=False, allow_nan=False), encoding="utf-8")
print(f"\nEO：solar_panel={n_eo_panel} crack={n_crack}（每張平均 {n_eo_panel / 696:.1f} 個模組）")
print(f"IR：solar_panel={n_ir_panel} hotspot={n_hot}（每張平均 {n_ir_panel / 696:.1f} 個模組）")
print("已存：", V5_PATH)
hot_rate = n_hot / max(n_hot + n_ir_panel, 1)
print(f"其中偽彩色 IR 圖的模組 {n_mod_col} 個，被標成 hotspot 的比例：{n_hot_col / max(n_mod_col, 1):.1%}"
      f"（非偽彩色：{(n_hot - n_hot_col) / max(n_hot + n_ir_panel - n_mod_col, 1):.1%}）")
print(f"IR 被標成 hotspot 的比例：{hot_rate:.1%}（訓練集約 1.7%）")
if hot_rate > 0.12:
    print("⚠ 比例偏高：HOT_TAU 可能太低，或測試 IR 圖的亮度分布和訓練不同（例如偽彩色）。"
          "建議把 HOT_K 改成 1 或調高 HOT_TAU（例如 1e-3）再產生，並看第 10 步的疊圖。")

r = subprocess.run([sys.executable, str(KIT_ROOT / "make_submission.py"), "--check", str(V5_PATH)],
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
print(r.stdout)


hotspot：每張圖取前 1 名且分數 >= 0.001 | EO crack：True（前 4%）
[IR] 8/696
[IR] 16/696
[IR] 24/696
[IR] 32/696
[IR] 40/696
[IR] 48/696
[IR] 56/696
[IR] 64/696
[IR] 72/696
[IR] 80/696
[IR] 88/696
[IR] 96/696
[IR] 104/696
[IR] 112/696
[IR] 120/696
[IR] 128/696
[IR] 136/696
[IR] 144/696
[IR] 152/696
[IR] 160/696
[IR] 168/696
[IR] 176/696
[IR] 184/696
[IR] 192/696
[IR] 200/696
[IR] 208/696
[IR] 216/696
[IR] 224/696
[IR] 232/696
[IR] 240/696
[IR] 248/696
[IR] 256/696
[IR] 264/696
[IR] 272/696
[IR] 280/696
[IR] 288/696
[IR] 296/696
[IR] 304/696
[IR] 312/696
[IR] 320/696
[IR] 328/696
[IR] 336/696
[IR] 344/696
[IR] 352/696
[IR] 360/696
[IR] 368/696
[IR] 376/696
[IR] 384/696
[IR] 392/696
[IR] 400/696
[IR] 408/696
[IR] 416/696
[IR] 424/696
[IR] 432/696
[IR] 440/696
[IR] 448/696
[IR] 456/696
[IR] 464/696
[IR] 472/696
[IR] 480/696
[IR] 488/696
[IR] 496/696
[IR] 504/696
[IR] 512/696
[IR] 520/696
[IR] 528/696
[IR] 536/696
[IR] 544/696
[IR] 552/696
[IR] 560/696
[IR] 568/696
[IR] 576/696
[IR] 584/696
[IR] 592/696

In [18]:
import json, numpy as np
from pathlib import Path
from shapely.geometry import Polygon

OUT = Path(r"C:\Users\Sharon\Downloads\VLPaicup\outputs")
A_path = OUT / "best_0.5476.json"   # 舊的最佳提交
B_path = OUT / "my_submission_v5.json"    # 新版

def load_hot(p):
    d = json.load(open(p, encoding="utf-8"))
    res = {}
    for k, v in d.items():
        objs = v if isinstance(v, list) else v.get("shapes", v.get("objects", []))
        res[k] = [Polygon(o["points"]).buffer(0) for o in objs
                  if o.get("label") == "hotspot" and len(o["points"]) >= 3]
    return res

def iou(a, b):
    u = a.union(b).area
    return a.intersection(b).area / u if u > 0 else 0

A, B = load_hot(A_path), load_hot(B_path)
na = sum(len(v) for v in A.values()); nb = sum(len(v) for v in B.values())
same = 0
for k in set(A) | set(B):
    used = set()
    for pa in A.get(k, []):
        for j, pb in enumerate(B.get(k, [])):
            if j not in used and iou(pa, pb) >= 0.5:
                used.add(j); same += 1; break
print(f"舊版 hotspot {na} 個，新版 {nb} 個，位置相同(IoU>=0.5)：{same} 個")
print(f"重疊比例：佔舊版 {same/max(na,1):.0%}，佔新版 {same/max(nb,1):.0%}")

舊版 hotspot 502 個，新版 510 個，位置相同(IoU>=0.5)：278 個
重疊比例：佔舊版 55%，佔新版 55%


In [20]:
import json
from pathlib import Path
from shapely.geometry import Polygon

OUT = Path(r"C:\Users\Sharon\Downloads\VLPaicup\outputs")
A = json.load(open(OUT / "best_0.5476.json", encoding="utf-8"))   # 舊版（固定門檻）
B = json.load(open(OUT / "v3_rank_k1.json", encoding="utf-8"))    # 新版（每張取前 1）

def poly(o):
    return Polygon(o["points"]).buffer(0)

def iou(a, b):
    u = a.union(b).area
    return a.intersection(b).area / u if u > 0 else 0

probe, kept, dropped = {}, 0, 0
for k, objs in B.items():
    hot_a = [poly(o) for o in A.get(k, []) if o.get("label") == "hotspot" and len(o["points"]) >= 3]
    new = []
    for o in objs:
        if o.get("label") == "hotspot":
            pb = poly(o)
            if any(iou(pb, pa) >= 0.5 for pa in hot_a):
                new.append(o); kept += 1
            else:
                dropped += 1
        else:
            new.append(o)          # solar_panel / crack 完全不動
    probe[k] = new

assert kept > 0, "沒有共同 hotspot，不能上傳"
json.dump(probe, open(OUT / "probe_shared.json", "w", encoding="utf-8"), ensure_ascii=False)
print(f"保留 hotspot {kept} 個，移除 {dropped} 個 → {OUT/'probe_shared.json'}")

保留 hotspot 278 個，移除 232 個 → C:\Users\Sharon\Downloads\VLPaicup\outputs\probe_shared.json
